In [1]:
import os
import requests
from typing import Dict, Any

USER_AGENT = os.getenv(
    "USER_AGENT",
    "TourGuideAI/1.0 (Learning project), Mozilla/5.0 (Windows NT 10.0; Win64; x64), Chrome/91.0.4472.124 Safari/537.36")

def search_place(query: str) -> Dict[str, Any]:
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": query,
        "format": "json",
        "limit": 1,
        "addressdetails": 1
    }
    headers = {"User-Agent": USER_AGENT}

    r = requests.get(url, params=params, headers=headers, timeout=10)
    r.raise_for_status()
    data = r.json()

    if not data:
        return {}

    return {
        "name": data[0]["display_name"],
        "lat": float(data[0]["lat"]),
        "lon": float(data[0]["lon"]),
        "city": data[0]["address"].get("city") or data[0]["address"].get("town"),
        "state": data[0]["address"].get("state"),
        "country": data[0]["address"].get("country")
    }


def get_museums_opening_hours(lat: float, lon: float, radius: int = 10000):

    query = f"""
    [out:json][timeout:25];

    (
      node["tourism"="museum"](around:{radius},{lat},{lon});
      way["tourism"="museum"](around:{radius},{lat},{lon});
      relation["tourism"="museum"](around:{radius},{lat},{lon});
    );

    out tags center;
    """

    r = requests.post(
        "https://overpass-api.de/api/interpreter",
        data=query,
        headers={"User-Agent": USER_AGENT},
        timeout=60
    )

    r.raise_for_status()
    data = r.json()

    museums = []

    for el in data.get("elements", []):
        tags = el.get("tags", {})

        if not tags.get("name"):
            continue

        lat_val = el.get("lat") or el.get("center", {}).get("lat")
        lon_val = el.get("lon") or el.get("center", {}).get("lon")

        museums.append({
            "name": tags.get("name"),
            "opening_hours": tags.get("opening_hours"),
            "lat": lat_val,
            "lon": lon_val,
            "street": tags.get("addr:street"),
            "housenumber": tags.get("addr:housenumber"),
            "postcode": tags.get("addr:postcode"),
            "city": tags.get("addr:city")
        })

    return museums

In [15]:
place = search_place("Raddusch, Brandenburg, Deutschland")

if place:
    museums = get_museums_opening_hours(place["lat"], place["lon"])

    for i, m in enumerate(museums, start=1):

        print( f" {i} Name:  {m['name']}--------------------")
        print( f"    Öffnungszeiten: {m.get('opening_hours') or 'Keine Informationen verfügbar'}")
        print( f"    Adresse: {m.get('street',' ')} {m.get('housenumber', ' ')}, {m.get('postcode', ' ')} {m.get('city', ' ')}")

 1 Name:  Gurkenmuseum--------------------
    Öffnungszeiten: Keine Informationen verfügbar
    Adresse: An der Dolzke 6, 03222 Lübbenau/Spreewald
 2 Name:  Heimatstube--------------------
    Öffnungszeiten: Keine Informationen verfügbar
    Adresse: None None, None None
 3 Name:  Haus für Mensch und Natur--------------------
    Öffnungszeiten: Tu-Su 10:00-17:00
    Adresse: Schulstraße 9, 03222 Lübbenau/Spreewald
 4 Name:  Mobile Welt des Ostens--------------------
    Öffnungszeiten: Keine Informationen verfügbar
    Adresse: None None, None None
 5 Name:  Heimatstube Burg (Spreewald)--------------------
    Öffnungszeiten: Tu-Su 12:00-17:00
    Adresse: None None, None None
 6 Name:  DDR-Museum "Haushalt"--------------------
    Öffnungszeiten: Keine Informationen verfügbar
    Adresse: None None, None None
 7 Name:  Spreewald-Museum Lübbenau--------------------
    Öffnungszeiten: Apr-Oct: Tu-Su,PH 10:30-18:00; Nov-Mar: Tu-Su,PH 11:00-16:00; Apr-Oct: Mo off; Nov-Mar: Mo off
    